In [71]:
# Movie Data Preprocessing
# Input:  data/movies_scraped.pkl
# Output: data/movies.pkl, data/similarity.pkl

import pandas as pd
import pickle
import string
from nltk.stem.snowball import SnowballStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

try:
    from nltk.corpus import stopwords
    stop_words = set(stopwords.words('english'))
except:
    import nltk
    nltk.download('stopwords')
    from nltk.corpus import stopwords
    stop_words = set(stopwords.words('english'))

stemmer = SnowballStemmer('english')
print('Imports ready')

Imports ready


## 1. Metadata

In [72]:
# Load scraped movie data
df = pd.read_pickle('data/movies_scraped.pkl')
print(f"Loaded {len(df)} movies")
df.head()

Loaded 13748 movies


,id,imdb_id,title,year,vote_average,vote_count,popularity,runtime,overview,tagline,poster_url,genres,actors,directors,keywords,watch_providers
0,663712,tt10403420,Terrifier 2,2022.0,6.692,2395,8.3584,138,"A year after the Miles County massacre, Art th...",Who's laughing now?,https://image.tmdb.org/t/p/w500//qEAlwXbYk6IHA...,"[Horror, Thriller]","[David Howard Thornton, Lauren LaVera, Elliott...",[Damien Leone],"[halloween, clown, gore, resurrection, sequel,...","{'US': {'flatrate': [{'id': 9, 'name': 'Amazon..."
1,9566,tt0116277,The Fan,1996.0,6.003,831,1.8023,116,When the San Francisco Giants pay center-field...,All fans have a favorite player. This one has...,https://image.tmdb.org/t/p/w500//lu7CjP8YES5dJ...,"[Thriller, Action, Drama]","[Robert De Niro, Wesley Snipes, Ellen Barkin, ...",[Tony Scott],"[sports, psychotic, luck, child custody, baseb...","{'US': {'flatrate': [], 'rent': [{'id': 10, 'n..."
2,4825,tt0048140,Guys and Dolls,1955.0,6.540,248,1.2118,149,"In New York, a gambler is challenged to take a...",It's a living breathing doll of a musical!,https://image.tmdb.org/t/p/w500//mrSM6laJJLBVd...,"[Comedy, Crime, Romance]","[Marlon Brando, Jean Simmons, Frank Sinatra, V...",[Joseph L. Mankiewicz],"[bet, new york city, gambling, missionary, mus...","{'US': {'flatrate': [{'id': 9, 'name': 'Amazon..."
3,212063,tt3089388,Tim's Vermeer,2013.0,7.252,163,1.3546,80,"Tim Jenison, a Texas based inventor, attempts ...",,https://image.tmdb.org/t/p/w500//vEF0f8dktdp4N...,[Documentary],"[Tim Jenison, Penn Jillette, Martin Mull, Tell...",[Teller],"[art collector, experiment, inventor, painting...","{'US': {'flatrate': [], 'rent': [{'id': 10, 'n..."
4,1537,tt0264472,Changing Lanes,2002.0,6.275,1168,5.2662,98,A rush-hour fender-bender on New York City's c...,One wrong turn deserves another.,https://image.tmdb.org/t/p/w500//8wh9WzTKo5pJw...,"[Thriller, Drama]","[Ben Affleck, Samuel L. Jackson, Toni Collette...",[Roger Michell],"[new york city, custody battle, revenge, lawye...","{'US': {'flatrate': [{'id': 257, 'name': 'fubo..."


In [73]:
# Ensure list columns are proper lists
for col in ['genres', 'actors', 'directors', 'keywords']:
    if col in df.columns:
        df[col] = df[col].apply(lambda x: x if isinstance(x, list) else [])
df['overview'] = df['overview'].fillna('')
df['tagline'] = df['tagline'].fillna('')
print('Data cleaned')

Data cleaned


In [74]:
# Process text and names
def process_text(text):
    if not text or not isinstance(text, str): return []
    text = text.lower()
    translator = str.maketrans('', '', string.punctuation)
    words = text.translate(translator).split()
    return [stemmer.stem(w) for w in words if w not in stop_words and len(w) > 2]

def process_names(names):
    if not isinstance(names, list): return []
    return [name.lower().replace(' ', '') for name in names if name]

df['overview_processed'] = df['overview'].apply(process_text)
df['tagline_processed'] = df['tagline'].apply(process_text)
df['actors_processed'] = df['actors'].apply(process_names)
df['directors_processed'] = df['directors'].apply(process_names)
df['genres_processed'] = df['genres'].apply(lambda x: [g.lower() for g in x] if isinstance(x, list) else [])
df['keywords_processed'] = df['keywords'].apply(lambda x: [stemmer.stem(k.lower().replace(' ', '')) for k in x] if isinstance(x, list) else [])
print('Features processed')

Features processed


In [75]:
# Create soup (combined features)
def create_soup(row):
    parts = []
    if pd.notna(row.get('vote_average')):
        parts.extend([f"rating{int(row['vote_average'])}"] * 2)
    parts.extend(row.get('genres_processed', []) * 2)
    parts.extend(row.get('actors_processed', []) * 2)
    parts.extend(row.get('directors_processed', []) * 2)
    parts.extend(row.get('keywords_processed', []))
    parts.extend(row.get('overview_processed', []))
    parts.extend(row.get('tagline_processed', []))
    return ' '.join(parts)

df['soup'] = df.apply(create_soup, axis=1)
print('Soup created')

Soup created


In [76]:
# TF-IDF
tf = TfidfVectorizer(analyzer='word', ngram_range=(1, 2), min_df=3, max_df=0.8, max_features=20000, stop_words='english', dtype='float32')
print('Fitting TF-IDF...')
tfidf_matrix = tf.fit_transform(df['soup'].values.astype(str))
print(f'TF-IDF shape: {tfidf_matrix.shape}')

Fitting TF-IDF...


C:\ProgramData\anaconda3\Lib\site-packages\sklearn\feature_extraction\text.py:2043: UserWarning: Only (<class 'numpy.float64'>, <class 'numpy.float32'>, <class 'numpy.float16'>) 'dtype' should be used. float32 'dtype' will be converted to np.float64.
  warnings.warn(


TF-IDF shape: (13748, 20000)


In [77]:
# Cosine similarity
print('Computing similarity...')
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
print(f'Similarity shape: {cosine_sim.shape}')

Computing similarity...
Similarity shape: (13748, 13748)


In [78]:
# Test recommendations
df = df.reset_index(drop=True)
indices = pd.Series(df.index, index=df['title'])

def get_recommendations(title, n=10):
    if title not in indices: return None
    idx = indices[title]
    if isinstance(idx, pd.Series): idx = idx.iloc[0]
    sim_scores = sorted(enumerate(cosine_sim[idx]), key=lambda x: x[1], reverse=True)[1:n+1]
    return df[['title', 'year', 'vote_average']].iloc[[i[0] for i in sim_scores]]

get_recommendations('The Dark Knight', 10)

,title,year,vote_average
3606,The Dark Knight Rises,2012.0,7.790
1070,Batman Begins,2005.0,7.720
7868,Batman,1989.0,7.237
3298,The Batman,2022.0,7.700
4030,The Prestige,2006.0,8.205
2211,Batman Forever,1995.0,5.449
13315,"Batman: The Long Halloween, Part Two",2021.0,7.402
1889,"Batman: The Long Halloween, Part One",2021.0,7.460
10039,Batman: Mask of the Phantasm,1993.0,7.487
5763,Following,1999.0,7.128


In [79]:
# Save output
with open('data/movies.pkl', 'wb') as f:
    pickle.dump(df, f)
with open('data/similarity.pkl', 'wb') as f:
    pickle.dump(cosine_sim, f)
print(f'Done! Saved {len(df)} movies.')

Done! Saved 13748 movies.


In [ ]:
# End of notebook - cells below are legacy

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

### 2. Credits

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

## 3. Keywords

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

## 4. Merging dataframes on id

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass